In [1]:
from automated_events_coding.pipeline import (
    step0_relevance,
    step1_events,
    step2_location,
    step3_event_lead,
    step4_interaction,
    step5_identification,
)
from automated_events_coding.pipeline.schemas import Event, EventList, EventLead, EventRecord, InteractionType
from automated_events_coding.llm.client import get_client
from automated_events_coding.llm.server import LlamaServerConfig, LlamaServer
from example_articles import ARTICLES, ARTICLES_BY_CATEGORY
from automated_events_coding.pipeline.prompts import STEP1_EVENT_EXTRACTION, STEP2_LOCATION_EXTRACTION, STEP3_EVENT_LEAD

import re
import json
from pydantic import TypeAdapter
from pprint import pprint


In [2]:
article_text = ARTICLES["010_E1"]
article_text

"ETAKA, 05 AUG (NAMPA) - Namibia Water Corporation (NamWater) on Saturday pumped water from Olushandja South wall outlet into the Etaka-Uuvudhiya earth canal, which stretches across 130 kilometers.The raw water is fit only for livestock consumption and agriculture purposes.NamWater Chief Executive Officer (CEO), Abraham Nehemia, said that the exercise was done due to previous and recurrent prevalent drought reported in the northern part of the country.Nehemia indicated that the Office of Prime Minister allocated N.dollar 3 million to ensure the completion of the project.'Households are expected to utilise the water for their daily needs, especially gardening for nutritional purposes,' he said.He added that they are confident that the water will be available in the dam to supply the canal until the rainy season.Nehemia also indicated that they are expecting the new pumps at Calueque to be commissioned soon, adding that NamWater is strategically looking into the rehabilitation of most of

In [3]:
relevance = step0_relevance.check_relevance(article_text=article_text)
relevance

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

RelevanceResult(relevant=True, meta={'relevance_label': 'Y', 'relevance_score': '0.6799'})

In [4]:
from automated_events_coding.pipeline.prompts import STEP1_EVENT_EXTRACTION, STEP2_LOCATION_EXTRACTION, STEP3_EVENT_LEAD
import re
import json
from pydantic import TypeAdapter
from pprint import pprint

json_pattern = r"\[.*?\]"
llama_server_config = LlamaServerConfig(
    model_path="/Users/pnadel01/models/gguf/unsloth_Qwen3.8-27B-GGUF/Qwen3.8-27B-UD-Q5_K_XL.gguf",
    temperature=0.7,
    top_p=0.80,
    top_k=20,
    extra_args=["--reasoning", "0"]
)

with LlamaServer(llama_server_config) as server:
    client = get_client(llama_server_config)

    # step 1
    response = client.chat.completions.create(
        model="Test",
        messages=[
            {"role": "system", "content": STEP1_EVENT_EXTRACTION},
            {"role": "user", "content": article_text},
        ],
    )
    raw = response.choices[0].message.content
    content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
    events = []
    for event in content:
        event_obj = Event(text=event["event_summary"])
        events.append(event_obj)
    validated = EventList(events=events)
    for i, event in enumerate(validated.events):
        print(f"Event {i+1}")
        pprint(event.text)
        print("="*20)

    # step 2
    placenames = []
    for i, event in enumerate(validated.events):
        response = client.chat.completions.create(
            model="Test",
            messages=[
                {"role": "system", "content": STEP2_LOCATION_EXTRACTION},
                {"role": "user", "content": event.text},
            ],
        )
        raw = response.choices[0].message.content
        content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
        placenames += content
    print("="*20)
    print("Placenames found:")
    print(placenames)
    validated_basins = step2_location.lookup_country_basin(placenames)
    print("Basins found:")
    print(validated_basins)
    print("="*20)

    # step 3
    print("="*20)
    event_template = "# Event: {event}\n# Article: {article}"
    event_led_extraction = []
    for i, event in enumerate(validated.events):
        response = client.chat.completions.create(
            model="Test",
            messages=[
                {"role": "system", "content": STEP3_EVENT_LEAD},
                {"role": "user", "content": event_template.format(event=event.text, article=article_text)},
            ],
        )
        raw = response.choices[0].message.content
        content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
        print(event.text)
        print(content)
        print("="*20)
        event_led_extraction += content


Event 1
('Namibia Water Corporation (NamWater) pumped water from the Olushandja South '
 'wall outlet into the Etaka-Uuvudhiya earth canal in Etaka.')
Event 2
('NamWater Chief Executive Officer Abraham Nehemia stated that the Office of '
 'Prime Minister allocated N.dollar 3 million to ensure the completion of the '
 'project.')
Event 3
('Omusati Governor Erginus Endjala urged community members to guard against '
 'the vandalism of the canal during an occasion at Etaka dam.')
Placenames found:
[{'placename': 'Namibia', 'placename_type': 'country'}, {'placename': 'Olushandja', 'placename_type': 'non-country'}, {'placename': 'Etaka-Uuvudhiya', 'placename_type': 'non-country'}, {'placename': 'Etaka', 'placename_type': 'non-country'}, {'placename': 'Namibia', 'placename_type': 'country'}, {'placename': 'Etaka dam', 'placename_type': 'non-country'}]
Basins found:
[CountryBasinLookup(bcode='ETOS', ccodes=['NAM', 'AGO'])]
Namibia Water Corporation (NamWater) pumped water from the Olushandja S

In [2]:
json_pattern = r"\[.*?\]"
llama_server_config = LlamaServerConfig(
    model_path="/Users/pnadel01/models/gguf/unsloth_Qwen3.8-27B-GGUF/Qwen3.8-27B-UD-Q5_K_XL.gguf",
    temperature=0.7,
    top_p=0.80,
    top_k=20,
    extra_args=["--reasoning", "0"]
)

def full_process(article_text, llama_server_config=llama_server_config):
    "Full process so far..."
    relevance = step0_relevance.check_relevance(article_text=article_text)
    if relevance.meta['relevance_label'] != 'Y':
        print("!! ARTICLE IRRELEVANT !!")

    event_template = "# Event: {event}\n# Article: {article}"
    with LlamaServer(llama_server_config) as server:
        client = get_client(llama_server_config)

        # step 1
        response = client.chat.completions.create(
            model="Test",
            messages=[
                {"role": "system", "content": STEP1_EVENT_EXTRACTION},
                {"role": "user", "content": article_text},
            ],
        )
        raw = response.choices[0].message.content
        content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
        events = []
        for event in content:
            event_obj = Event(text=event["event_summary"])
            events.append(event_obj)
        validated = EventList(events=events)
        for i, event in enumerate(validated.events):
            print(f"Event {i+1}")
            pprint(event.text)
            print("="*20)

        # step 2
        placenames = []
        for i, event in enumerate(validated.events):
            response = client.chat.completions.create(
                model="Test",
                messages=[
                    {"role": "system", "content": STEP2_LOCATION_EXTRACTION},
                    {"role": "user", "content": event_template.format(event=event.text, article=article_text)},
                ],
            )
            raw = response.choices[0].message.content
            content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
            placenames += content
        print("="*20)
        print("Placenames found:")
        print(placenames)
        validated_basins = step2_location.lookup_country_basin(placenames)
        print("Basins found:")
        print(validated_basins)
        print("="*20)

        # step 3
        print("="*20)
        event_led_extraction = []
        for i, event in enumerate(validated.events):
            response = client.chat.completions.create(
                model="Test",
                messages=[
                    {"role": "system", "content": STEP3_EVENT_LEAD},
                    {"role": "user", "content": event_template.format(event=event.text, article=article_text)},
                ],
            )
            raw = response.choices[0].message.content
            content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
            pprint(event.text)
            print(content)
            print("="*20)
            event_led_extraction += content

    return validated, validated_basins, event_led_extraction

In [3]:
article_text = ARTICLES["028_E1"]
validated, validated_basins, event_led_extraction = full_process(article_text)

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

Event 1
('The Mozambican government approved a National Water Resource Management Plan '
 'for the next 20 years, estimated to cost 28 billion US dollars, to assess '
 'water availability and prioritize construction of storage infrastructures '
 'like dams at Mpanda Nkua and Mapai.')
Event 2
('The Mozambican government announced its intention to invest over 300,000 '
 'dollars over the next seven years in implementing the Fourth National Plan '
 'for the Advancement of Women to promote gender equality and coordinate '
 "inter-sector activities for women's empowerment.")
Placenames found:
[{'placename': 'Mozambique', 'placename_type': 'country'}, {'placename': 'Mpanda Nkua', 'placename_type': 'non-country'}, {'placename': 'Mapai', 'placename_type': 'non-country'}, {'placename': 'Mozambique', 'placename_type': 'country'}]
Basins found:
[CountryBasinLookup(bcode='BUZI', ccodes=[]), CountryBasinLookup(bcode='ICMT', ccodes=[]), CountryBasinLookup(bcode='LKCW', ccodes=[]), CountryBasinLookup

In [4]:
article_text = ARTICLES["018_E1"]
validated, validated_basins, event_led_extraction = full_process(article_text)

Event 1
('The UNESCO Regional Office for Southern Africa issued a news release '
 "announcing that it will hold an inception workshop for the project 'A "
 'community focused flood Early Warning System for the Buzi-Pungwe and Save '
 "(BuPuSa) Transboundary River Basins' in Maputo, Mozambique on 18 May 2022.")
Placenames found:
[{'placename': 'Mozambique', 'placename_type': 'country'}, {'placename': 'Maputo', 'placename_type': 'non-country'}, {'placename': 'Buzi', 'placename_type': 'non-country'}, {'placename': 'Pungwe', 'placename_type': 'non-country'}, {'placename': 'Save', 'placename_type': 'non-country'}]
Basins found:
[CountryBasinLookup(bcode='MPUT', ccodes=['SWZ', 'MOZ', 'ZAF', 'ZWE']), CountryBasinLookup(bcode='BUZI', ccodes=['SWZ', 'MOZ', 'ZAF', 'ZWE']), CountryBasinLookup(bcode='PUNG', ccodes=['SWZ', 'MOZ', 'ZAF', 'ZWE']), CountryBasinLookup(bcode='SABI', ccodes=['SWZ', 'MOZ', 'ZAF', 'ZWE'])]
('The UNESCO Regional Office for Southern Africa issued a news release '
 "announcin

In [5]:
article_text = ARTICLES["065_E1"]
validated, validated_basins, event_led_extraction = full_process(article_text)

Event 1
('Chile asked the International Court of Justice (ICJ) for an extra three '
 'months to provide a written response to Bolivian reports in the Silala River '
 'case.')
Event 2
("Diego Pary, Bolivia's Foreign Affairs Minister, stated that Bolivia will "
 'demand the right to reply if the ICJ allows Chile to present a new report.')
Event 3
('Bolivia presented a counter claim to defend the sovereignty of Silala '
 'waters, located in Quetena Chico canton, Potosi department.')
Placenames found:
[{'placename': 'Chile', 'placename_type': 'country'}, {'placename': 'Bolivia', 'placename_type': 'country'}, {'placename': 'Silala River', 'placename_type': 'non-country'}, {'placename': 'Quetena Chico', 'placename_type': 'non-country'}, {'placename': 'Potosi', 'placename_type': 'non-country'}, {'placename': 'Bolivia', 'placename_type': 'country'}, {'placename': 'Chile', 'placename_type': 'country'}, {'placename': 'Silala River', 'placename_type': 'non-country'}, {'placename': 'Bolivia', 'pla